In [ ]:
import os
import numpy as np
import random
import argparse
import torch
from config import FLConfig
from data_loader import load_sent140_partitions
from server import Server

def set_seed(seed=42):
    """Fixa todas as seeds para garantir reprodutibilidade total."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    os.environ['PYTHONHASHSEED'] = str(seed)


def save_accuracy_history(acc_history, algorithm, mu, straggler_rate, out_dir="acc"):
    os.makedirs(out_dir, exist_ok=True)
    if mu == None:
        filename = f"acc_algorithm={algorithm}_StragglerRate={straggler_rate}.npy"
    else:
        filename = f"acc_algorithm={algorithm}_mu={mu}_StragglerRate={straggler_rate}.npy"
        
    path = os.path.join(out_dir, filename)

    np.save(path, np.array(acc_history))
    print(f"Saved accuracy history to {path}")

def save_loss_history(loss_history, algorithm, mu, straggler_rate, out_dir="loss"):
    os.makedirs(out_dir, exist_ok=True)
    if mu == None:
        filename = f"loss_algorithm={algorithm}_StragglerRate={straggler_rate}.npy" 
    else:
        filename = f"loss_algorithm={algorithm}_mu={mu}_StragglerRate={straggler_rate}.npy"
        
    path = os.path.join(out_dir, filename)

    np.save(path, np.array(loss_history))
    print(f"Saved loss history to {path}")

def save_loss_test(loss_history, algorithm, mu, straggler_rate, out_dir="loss test"):
    os.makedirs(out_dir, exist_ok=True)
    if mu == None:
        filename = f"loss_algorithm={algorithm}_StragglerRate={straggler_rate}.npy" 
    else:
        filename = f"loss_algorithm={algorithm}_mu={mu}_StragglerRate={straggler_rate}.npy"
        
    path = os.path.join(out_dir, filename)

    np.save(path, np.array(loss_history))
    print(f"Saved loss history to {path}")
    
def main():
    E = 20
    B = 10
    iid = False
    DATASET = "sent140"
    algorithms = ["fedavg", "fedprox"]
    mus = [0.0, 0.01]
    straggler_rate = [0.0, 0.5, 0.9]
    global_acc_history = {}
    global_loss_history = {}
    set_seed(42)
    config = FLConfig(dataset=DATASET)
    train_loaders, test_loader = load_sent140_partitions(config)
    for stglr in straggler_rate:
        for algorithm in algorithms:
            set_seed(42)
            if algorithm == "fedavg":
                mu = None
                config.straggler_rate = stglr
                config.algorithm = algorithm
                config.mu = mu
                #train_loaders, test_loader = load_mnist_partitions(config)
                device = torch.device("cuda" if (config.use_cuda and torch.cuda.is_available()) else "cpu")
                print(f"\nRunning experiment: iid={iid}, mu={mu}, straggler_rate={stglr}, algorithm={algorithm} ")
            
                # Initialize and run federated server
                server = Server(config, train_loaders, test_loader, device)
                server.run()
                     
                
                #saving accuracy and loss history
                save_accuracy_history(server.accuracy_history, algorithm, mu, stglr)
                save_loss_history(server.loss_history, algorithm, mu, stglr)
                save_loss_test(server.loss_test, algorithm, mu, stglr)
                
                # Ensure save directory exists
                save_path = config.save_path.format(algorithm=algorithm)
                os.makedirs(os.path.dirname(save_path), exist_ok=True)
            
                # Save global model
                torch.save(server.global_model.state_dict(), save_path)
                print(f"Saved global model to {save_path}")
                
            else:    
                for mu in mus:
                    set_seed(42)
                    config.straggler_rate = stglr
                    config.algorithm = algorithm
                    config.mu = mu
                    #train_loaders, test_loader = load_mnist_partitions(config)
                    device = torch.device("cuda" if (config.use_cuda and torch.cuda.is_available()) else "cpu")
                    print(f"\nRunning experiment: iid={iid}, mu={mu}, straggler_rate={stglr}, algorithm={algorithm} ")
                    
                    # Initialize and run federated server
                    server = Server(config, train_loaders, test_loader, device)
                    server.run()
                          
                    #saving accuracy and loss history
                    save_accuracy_history(server.accuracy_history, algorithm, mu, stglr)
                    save_loss_history(server.loss_history, algorithm, mu, stglr)
                    save_loss_test(server.loss_test, algorithm, mu, stglr)
                    
                    # Ensure save directory exists
                    save_path = config.save_path.format(algorithm=algorithm)
                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                
                    # Save global model
                    torch.save(server.global_model.state_dict(), save_path)
                    print(f"Saved global model to {save_path}")
        

if __name__ == "__main__":
    main()

Lendo Sent140 pré-processado pelo LEAF...
[Pré-seleção Sent140] clientes=772, total=41784, média=54.1, std=31.5, min=31, max=281
Carregando GloVe filtrado para o vocabulário do Sent140...
[Sent140] Partição concluída.
  Clientes: 772
  Total de amostras: 41784
  Tamanho por cliente -> min: 31, max: 281, média: 54.1, std: 31.5
  Treino total: 33101 | Teste total: 8683
  Vocabulário: 37754 | Tokens com GloVe encontrado: 17425

Running experiment: iid=False, mu=None, straggler_rate=0.0, algorithm=fedavg 


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_paper_style():
    straggler_rates = [0.0, 0.5, 0.9]
    
    # Configurações de estilo idênticas ao paper
    styles = {
        'fedavg':      {'color': 'red',   'label': 'FedAvg',           'linestyle': '--', 'linewidth': 2},
        'fedprox_mu0': {'color': 'blue',  'label': r'FedProx ($\mu=0$)', 'linestyle': ':',  'linewidth': 2},
        'fedprox_mu_high': {'color': 'green', 'label': r'FedProx ($\mu=1.0$)', 'linestyle': '-',  'linewidth': 2}
    }

    for stglr in straggler_rates:
        plt.figure(figsize=(6, 4))
        
        loss_avg = np.load(f"loss test/loss_algorithm=fedavg_StragglerRate={stglr}.npy")
        loss_prox0 = np.load(f"loss test/loss_algorithm=fedprox_mu=0.0_StragglerRate={stglr}.npy")
        loss_prox_high = np.load(f"loss test/loss_algorithm=fedprox_mu=1.0_StragglerRate={stglr}.npy")
      
        plt.plot(loss_avg, **styles['fedavg'])
        plt.plot(loss_prox0, **styles['fedprox_mu0'])
        plt.plot(loss_prox_high, **styles['fedprox_mu_high'])
       
        plt.title(f"SENT140 - (Stragglers {int(stglr*100)}%)", fontsize=12)
        plt.xlabel("Communication Rounds", fontsize=10)
        plt.ylabel("Test Loss", fontsize=10)
        
        plt.ylim(1.0, 4.0) 
        
        plt.legend(loc='upper right')
        plt.grid(True, which='both', linestyle='--', alpha=0.3)
         
        plt.gca().spines['top'].set_visible(False)
        plt.gca().spines['right'].set_visible(False)
        
        plt.tight_layout()
        plt.savefig(f"fedprox SENT140 straggler rate={stglr} testset.pdf", dpi=300, bbox_inches='tight')
        plt.show()

plot_paper_style()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_paper_style():
    straggler_rates = [0.0, 0.5, 0.9]
    
    # Configurações de estilo idênticas ao paper
    styles = {
        'fedavg':      {'color': 'red',   'label': 'FedAvg',           'linestyle': '--', 'linewidth': 2},
        'fedprox_mu0': {'color': 'blue',  'label': r'FedProx ($\mu=0$)', 'linestyle': ':',  'linewidth': 2},
        'fedprox_mu_high': {'color': 'green', 'label': r'FedProx ($\mu=1.0$)', 'linestyle': '-',  'linewidth': 2}
    }

    for stglr in straggler_rates:
        plt.figure(figsize=(6, 4))
        
        loss_avg = np.load(f"loss/loss_algorithm=fedavg_StragglerRate={stglr}.npy")
        loss_prox0 = np.load(f"loss/loss_algorithm=fedprox_mu=0.0_StragglerRate={stglr}.npy")
        loss_prox_high = np.load(f"loss/loss_algorithm=fedprox_mu=1.0_StragglerRate={stglr}.npy")
      
        plt.plot(loss_avg, **styles['fedavg'])
        plt.plot(loss_prox0, **styles['fedprox_mu0'])
        plt.plot(loss_prox_high, **styles['fedprox_mu_high'])
       
        plt.title(f"SENT140 - (Stragglers {int(stglr*100)}%)", fontsize=12)
        plt.xlabel("Communication Rounds", fontsize=10)
        plt.ylabel("Training Loss", fontsize=10)
        
        plt.ylim(1.0, 4.0) 
        
        plt.legend(loc='upper right')
        plt.grid(True, which='both', linestyle='--', alpha=0.3)
         
        plt.gca().spines['top'].set_visible(False)
        plt.gca().spines['right'].set_visible(False)
        
        plt.tight_layout()
        plt.savefig(f"fedprox SENT140 straggler rate={stglr} trainset.pdf", dpi=300, bbox_inches='tight')
        plt.show()

plot_paper_style()
